In [1]:
import pandas as pd
from parepy_toolbox import *
import openpyxl
pd.set_option('display.max_columns', None)
import numpy as np
from dissertacao import meu_problema_murilo

In [2]:
m_rd = 29.36
gamma_g = 1.40
gamma_q = 1.40
chi_list = [0.1, 0.15, 0.3, 0.45, 0.60]
df_data = []

dados_viga = {'d_b (m)': 8/1000, 'd_linha (m)': 3.9/100,'n_b': 3, 'gamma_c': 1.00, 'gamma_s': 1.00, 'b_w (m)': 0.2, 'h (m)': 0.50, 'cob (m)': 0.025, 'ano_construcao': 2000}
dados_corrosao = {'k_c': 30.5, 'k_fc': 1.7, 'a_d': 0, 'k_ad': 0.32, 'k_co2': 15.5, 'k_rh': 1300, 'k_ce': 1.3}
none_variable = {'dados_viga': dados_viga, 'tempos reais': list(range(0, 101, 1)), 'dados_corrosao': dados_corrosao}


for id, chi in enumerate(chi_list):


    den_g = gamma_g + gamma_q*chi/(1-chi)
    den_q = gamma_g*(1-chi)/chi + gamma_q
    m_gk = m_rd/den_g
    m_qk = m_rd/den_q

    # Data
    g = {'type': 'normal', 'loc': 1.06*m_gk, 'scale': 0.12*1.06*m_gk, 'stochastic variable': False, 'seed': None}
    q = {'type': 'gumbel max', 'loc': 0.21*m_qk, 'scale': 0.21*0.76*m_qk, 'stochastic variable': True, 'seed': None}
    f_ck = {'type': 'normal', 'loc': 1.22*25000, 'scale': 0.15*1.22*25000, 'stochastic variable': False, 'seed': None}
    f_yk = {'type': 'normal', 'loc': 1.22*500000, 'scale': 0.04*1.22*500000, 'stochastic variable': False, 'seed': None}
    temp = {'type': 'normal', 'loc': 21.10, 'scale': 0.56, 'stochastic variable': True, 'seed': None}
    u_r = {'type': 'BETA', 'a': 2.938871209812339, 'b': 2.209857158984442, 'loc': 57.250522472205915, 'scale': 14.955723347873345, 'stochastic variable': True, 'seed': None}
    i_corr_20 = {'type': 'lognormal', 'loc': 0.431, 'scale': 0.259, 'stochastic variable': False, 'seed': None}
    teta_r = {'type': 'lognormal', 'loc': 1, 'scale': 0.05, 'stochastic variable': False, 'seed': None}
    teta_s = {'type': 'lognormal', 'loc': 1, 'scale': 0.05, 'stochastic variable': False, 'seed': None}
    var = [g, q, f_ck, f_yk, temp, u_r, i_corr_20, teta_r, teta_s]

    # PAREpy setup
    setup = {
                'number of samples': 100000, 
                'number of dimensions': len(var), 
                'numerical model': {'model sampling': 'mcs-time', 'time steps': len(none_variable['tempos reais'])}, 
                'variables settings': var, 
                'number of state limit functions or constraints': 7, 
                'none variable': none_variable,
                'objective function': meu_problema_murilo,
                'type process': 'auto',
                'name simulation': 'exemplo_dissertação_2936_250k',
            }
    # Call algorithm
    results, pf, beta = sampling_algorithm_structural_analysis(setup)

    # Assembly results
    dic = {}
    dic['chi'] = [chi] * len(none_variable['tempos reais'])
    dic['time'] = none_variable['tempos reais']
    dic['pf'] = pf[0]
    dic['beta'] = beta[0]
    data = pd.DataFrame(dic, columns=['chi', 'time', 'pf', 'beta'])
    df_data.append(data)

PARE^py Report: 

- Output file name: exemplo_dissertação_2936_250k_MCS-TIME_20240606-140055.txt
- Processing time (s): 6017.467382669449  (serial kernel)
PARE^py Report: 

- Output file name: exemplo_dissertação_2936_250k_MCS-TIME_20240606-152503.txt
- Processing time (s): 5048.400757074356  (serial kernel)
PARE^py Report: 

- Output file name: exemplo_dissertação_2936_250k_MCS-TIME_20240606-164138.txt
- Processing time (s): 4614.481696844101  (serial kernel)
PARE^py Report: 

- Output file name: exemplo_dissertação_2936_250k_MCS-TIME_20240606-180609.txt
- Processing time (s): 5113.317326784134  (serial kernel)
PARE^py Report: 

- Output file name: exemplo_dissertação_2936_250k_MCS-TIME_20240606-194301.txt
- Processing time (s): 5738.01558470726  (serial kernel)


In [4]:
# Write in excel file
with pd.ExcelWriter('dataset_2936.xlsx', engine='openpyxl') as writer:
    # Loop através de cada categoria e dados correspondentes
    for i, value in enumerate(chi_list):
        # Salvando o DataFrame na aba correspondente
        aux = df_data[i].copy()
        aux.to_excel(writer, sheet_name=str(value))